# AgentComet - Complete Feature Guide

This notebook demonstrates **all** features of AgentComet:

1. **UAFAgent** - Load and run .uaf agents with simplified API
2. **Memory Management** - Conversation history with full/limited modes
3. **LLM Integration** - Inject any LLM provider
4. **Orchestration** - Connect multiple agents
5. **Workflow Templates** - Pipeline, Fan-Out/Fan-In, Map-Reduce
6. **Conditional Routing** - Dynamic agent selection
7. **Hot Reloading** - Update agents without restart
8. **Version Control** - Git-like VCS for .uaf files
9. **Message Broker** - Inter-agent communication

**Prerequisites:**
```bash
pip install uaf_compiler langchain-ollama langchain-core
pip install -e .  # Install agentcomet
```

In [ ]:
import os
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

# AgentComet imports
from agentcomet.agents import UAFAgent, AgentResponse
from agentcomet.orchestrators import AgentOrchestrator
from agentcomet.workflows import WorkflowBuilder, WorkflowTemplates
from agentcomet.vcs import Repository
from agentcomet.communication import MessageBroker, Message

# Agent path
AGENT_PATH = "dummy_agents/mathematics-reasoning-agent.uaf"
print(f"Agent exists: {os.path.exists(AGENT_PATH)}")

In [ ]:
# Configure LLM (used throughout the notebook)
llm = ChatOllama(
    base_url="http://localhost:11434",
    model="gemma3:4b",
    temperature=0
)
print("LLM configured:", llm.model)

---

## 1. UAFAgent - Basic Usage

Load a `.uaf` agent and invoke it with plain text input.

In [ ]:
# Create an agent
agent = UAFAgent(
    name="math_agent",
    uaf_path=AGENT_PATH,
    llm=llm
)

# Invoke with plain text (no message wrapping needed!)
response = agent.invoke("What is 25 * 4?")

# Access the response
print("Content:", response.content)     # Plain text output
print("Type:", type(response))          # AgentResponse
print("Raw keys:", response.raw.keys()) # Full state dict

agent.cleanup()

### AgentResponse Object

Every `invoke()` returns an `AgentResponse` with:
- `.content` - Plain text output (string)
- `.messages` - Full message history (list)
- `.raw` - Original dict response from the agent

In [ ]:
agent = UAFAgent("demo", AGENT_PATH, llm=llm)
response = agent.invoke("What is the derivative of x^2?")

# String conversion returns content
print(str(response))

# Check message count
print(f"\nMessages in response: {len(response.messages)}")

agent.cleanup()

---

## 2. Memory Management

Agents can maintain conversation history for multi-turn interactions.

### Memory Modes:
- `memory=None` (default) - No memory, each call is independent
- `memory="full"` - Keep all messages forever
- `memory=N` - Keep only the last N messages (sliding window)

In [ ]:
# Agent with full memory
agent = UAFAgent("conversation", AGENT_PATH, llm=llm, memory="full")

# Multi-turn conversation
r1 = agent.invoke("What is 10 + 5?")
print("Q1:", r1.content)

# Follow-up question (agent remembers context)
r2 = agent.invoke("Double that result")
print("Q2:", r2.content)

r3 = agent.invoke("Now subtract 10")
print("Q3:", r3.content)

# Check history
print(f"\nTotal messages in history: {len(agent.get_history())}")

### Memory Methods

In [ ]:
# get_history() - View current message history
history = agent.get_history()
print(f"History length: {len(history)}")
for i, msg in enumerate(history):
    print(f"  [{i}] {type(msg).__name__}: {str(msg.content)[:50]}...")

# clear_memory() - Reset conversation
agent.clear_memory()
print(f"\nAfter clear: {len(agent.get_history())} messages")

agent.cleanup()

### Limited Memory (Sliding Window)

In [ ]:
# Keep only last 4 messages
agent = UAFAgent("limited", AGENT_PATH, llm=llm, memory=4)

# Ask 6 questions
questions = ["1+1?", "2+2?", "3+3?", "4+4?", "5+5?", "6+6?"]
for q in questions:
    r = agent.invoke(q)
    print(f"{q} -> {r.content[:20]}...")

# History should only have last 4
print(f"\nHistory size: {len(agent.get_history())} (limited to 4)")

agent.cleanup()

---

## 3. Orchestration - Connecting Multiple Agents

Use `AgentOrchestrator` to build and run multi-agent workflows.

In [ ]:
# Create orchestrator with global LLM
orch = AgentOrchestrator(llm=llm)

# Add agents
orch.add_agent('analyzer', AGENT_PATH)
orch.add_agent('solver', AGENT_PATH)
orch.add_agent('verifier', AGENT_PATH)

# Connect them: analyzer -> solver -> verifier
orch.connect('analyzer', 'solver')
orch.connect('solver', 'verifier')

# Run the workflow
result = orch.run({'messages': [HumanMessage(content='Calculate 15% of 200')]})
print("Workflow result keys:", list(result.keys()))

### Per-Agent LLM Configuration

Different agents can use different LLMs.

In [ ]:
# Create a second LLM config (example - same model for demo)
llm_fast = ChatOllama(base_url="http://localhost:11434", model="gemma3:4b", temperature=0.5)

# Orchestrator with mixed LLMs
orch = AgentOrchestrator(llm=llm)  # Default LLM
orch.add_agent('agent_a', AGENT_PATH)                    # Uses default
orch.add_agent('agent_b', AGENT_PATH, llm=llm_fast)      # Uses llm_fast
orch.connect('agent_a', 'agent_b')

print("agent_a uses: default (llm)")
print("agent_b uses: llm_fast")

---

## 4. Workflow Templates

Pre-built patterns for common workflow structures.

### 4.1 Pipeline Template

Linear chain: A → B → C

In [ ]:
# Create pipeline workflow
workflow = WorkflowTemplates.pipeline({
    'step1': AGENT_PATH,
    'step2': AGENT_PATH,
    'step3': AGENT_PATH
})

# Run with orchestrator
orch = AgentOrchestrator(workflow, llm=llm)
result = orch.run({'messages': [HumanMessage(content='What is 7 * 8?')]})
print("Pipeline complete")

### 4.2 Fan-Out/Fan-In Template

Parallel execution: Start → [P1, P2, P3] → End

In [ ]:
# Fan-out/Fan-in workflow
workflow = WorkflowTemplates.fan_out_fan_in(
    start_agent={'distributor': AGENT_PATH},
    parallel_agents={
        'worker1': AGENT_PATH,
        'worker2': AGENT_PATH,
        'worker3': AGENT_PATH
    },
    end_agent={'aggregator': AGENT_PATH}
)

print("Fan-out/Fan-in workflow created")
print(f"Agents: {list(workflow.agents.keys())}")

### 4.3 Map-Reduce Template

Alias for Fan-Out/Fan-In pattern.

In [ ]:
workflow = WorkflowTemplates.map_reduce(
    mapper={'mapper': AGENT_PATH},
    workers={'w1': AGENT_PATH, 'w2': AGENT_PATH},
    reducer={'reducer': AGENT_PATH}
)
print(f"Map-Reduce agents: {list(workflow.agents.keys())}")

---

## 5. WorkflowBuilder - Custom Workflows

Build complex workflows with fine-grained control.

In [ ]:
# Create workflow builder with default LLM
builder = WorkflowBuilder(default_llm=llm)

# Add agents
builder.add_agent('intake', AGENT_PATH)
builder.add_agent('processor', AGENT_PATH)
builder.add_agent('validator', AGENT_PATH)
builder.add_agent('reporter', AGENT_PATH)

# Define connections
builder.connect('intake', 'processor')
builder.connect('processor', 'validator')
builder.connect('validator', 'reporter')

# View the workflow
print("Agents:", list(builder.agents.keys()))
print("DAG edges:", builder.dag.get_edges() if hasattr(builder.dag, 'get_edges') else 'N/A')

### Parallel Execution Block

In [ ]:
builder = WorkflowBuilder(default_llm=llm)

builder.add_agent('start', AGENT_PATH)
builder.add_agent('parallel_a', AGENT_PATH)
builder.add_agent('parallel_b', AGENT_PATH)
builder.add_agent('end', AGENT_PATH)

# Define parallel group
builder.parallel(['parallel_a', 'parallel_b'], merge_strategy='consensus')

print(f"Parallel groups: {builder.parallel_groups}")

### Conditional Routing

In [ ]:
builder = WorkflowBuilder(default_llm=llm)

builder.add_agent('classifier', AGENT_PATH)
builder.add_agent('easy_handler', AGENT_PATH)
builder.add_agent('hard_handler', AGENT_PATH)
builder.add_agent('expert_handler', AGENT_PATH)

# Route based on classifier output
builder.route(
    from_agent='classifier',
    conditions={
        'easy': 'easy_handler',
        'hard': 'hard_handler',
        'expert': 'expert_handler'
    }
)

print(f"Routers defined: {len(builder.routers)}")

---

## 6. Hot Reloading

Reload agent logic without restarting Python. Memory is preserved!

In [ ]:
agent = UAFAgent("reloadable", AGENT_PATH, llm=llm, memory="full")

# First interaction
r1 = agent.invoke("Remember this number: 42")
print("Before reload:", r1.content[:100])

# Simulate agent update (in real use, you'd .rebuild() the .uaf)
print("\n--- Reloading agent ---")
agent.reload()

# Memory preserved after reload
r2 = agent.invoke("What number did I mention?")
print("After reload:", r2.content[:100])

print(f"\nHistory preserved: {len(agent.get_history())} messages")
agent.cleanup()

---

## 7. Version Control System (VCS)

Git-like version control specifically for .uaf agent files.

### Repository Methods:
- `init()` - Initialize a new repository
- `commit(message, files)` - Commit agent files
- `log()` - View commit history

In [ ]:
# Initialize repository
repo = Repository(os.getcwd())
repo.init()  # Creates .agentcomet directory

In [ ]:
# Commit an agent
try:
    commit_hash = repo.commit(
        message="Add math reasoning agent v1.0",
        files=[AGENT_PATH],
        author="Developer"
    )
    print(f"Committed: {commit_hash[:12]}...")
except Exception as e:
    print(f"Commit error: {e}")

In [ ]:
# Make another commit
try:
    repo.commit(
        message="Update math agent with improved prompts",
        files=[AGENT_PATH],
        author="Developer"
    )
except Exception as e:
    print(f"Commit error: {e}")

In [ ]:
# View commit history
print("=" * 50)
print("COMMIT HISTORY")
print("=" * 50)
repo.log()

---

## 8. Message Broker (Inter-Agent Communication)

Experimental feature for agents to communicate directly.

In [ ]:
# Create a message broker
broker = MessageBroker()

# Send a message
msg = Message(
    sender="agent_a",
    receiver="agent_b",
    content={"data": "Hello from Agent A!", "priority": "high"}
)
broker.send(msg)
print(f"Sent message from {msg.sender} to {msg.receiver}")

In [ ]:
# Receive messages
received = broker.receive("agent_b")
if received:
    print(f"Agent B received: {received.content}")
else:
    print("No messages for agent_b")

In [ ]:
# Subscribe to messages (callback-based)
def on_message(msg):
    print(f"Callback: Got message! Content: {msg.content}")

broker.subscribe("agent_c", on_message)

# Send to subscribed agent - callback fires immediately
broker.send(Message(sender="system", receiver="agent_c", content={"alert": "New task!"}))

In [ ]:
# Get all pending messages at once
broker.send(Message(sender="x", receiver="agent_d", content={"n": 1}))
broker.send(Message(sender="y", receiver="agent_d", content={"n": 2}))
broker.send(Message(sender="z", receiver="agent_d", content={"n": 3}))

all_msgs = broker.get_all_messages("agent_d")
print(f"Agent D got {len(all_msgs)} messages")
for m in all_msgs:
    print(f"  From {m.sender}: {m.content}")

---

## 9. CLI Commands Reference

AgentComet includes a CLI tool `afc` for common operations:

```bash
# Initialize new agent project
afc init my_agent

# Build agent into .uaf
afc build --setup uaf_setup.yaml

# Run an agent
afc run agent.uaf

# Version control
afc vcs init
afc vcs commit -m "message" file.uaf
afc vcs log
```

---

## Summary - All AgentComet Features

| Feature | Class/Method | Description |
|---------|--------------|-------------|
| Load Agent | `UAFAgent(name, path, llm, memory)` | Load .uaf with LLM and memory |
| Invoke | `agent.invoke("text")` | Run with plain text input |
| Response | `response.content` | Get plain text output |
| Memory | `memory="full"` or `memory=5` | Full or sliding window |
| History | `agent.get_history()` | Get message history |
| Clear | `agent.clear_memory()` | Reset conversation |
| Reload | `agent.reload()` | Hot-reload agent logic |
| Orchestrate | `AgentOrchestrator(llm=llm)` | Multi-agent workflows |
| Add Agent | `orch.add_agent(name, path, llm)` | Add to workflow |
| Connect | `orch.connect(a, b)` | Link agents |
| Pipeline | `WorkflowTemplates.pipeline({})` | Linear chain |
| Fan-Out | `WorkflowTemplates.fan_out_fan_in()` | Parallel execution |
| Route | `builder.route(from, conditions)` | Conditional paths |
| VCS Init | `repo.init()` | Create repository |
| Commit | `repo.commit(msg, files)` | Version agent |
| Log | `repo.log()` | View history |
| Message | `broker.send(Message(...))` | Inter-agent messaging |